In [1]:
import sys

print(sys.executable)

c:\Users\RIDDHI ASHAR\Desktop\financial-intelligence-engine\venv\Scripts\python.exe


In [1]:
import sys

print(sys.executable)

c:\Users\RIDDHI ASHAR\Desktop\music streaming\music_project\venv\Scripts\python.exe


In [2]:
import pandas as pd
import sqlite3
import os
import csv

EVENTS_FILE = "userid-timestamp-artid-artname-traid-traname.tsv"
PROFILE_FILE = "userid-profile.tsv"
DB_FILE = "lastfm_streaming.db"

for f in [EVENTS_FILE, PROFILE_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Can't find {f} — make sure your terminal's cwd is the project folder.")
    size_mb = os.path.getsize(f) / 1_000_000
    print(f"{f}: {size_mb:,.1f} MB")

userid-timestamp-artid-artname-traid-traname.tsv: 2,529.2 MB
userid-profile.tsv: 0.0 MB


In [3]:
with open(PROFILE_FILE, encoding="utf-8", errors="replace") as f:
    first_line = f.readline()

first_token = first_line.split("\t")[0].strip().lower()
has_header = not first_token.startswith("user_")
print("Detected header row:", has_header, "| first value on line 1 was:", repr(first_token))

profile_cols = ["user_id", "gender", "age", "country", "signup_date"]
users = pd.read_csv(
    PROFILE_FILE,
    sep="\t",
    header=0 if has_header else None,
    names=profile_cols,
    dtype=str,
    encoding="utf-8",
)

print("Shape:", users.shape)
users.head()

Detected header row: True | first value on line 1 was: '#id'
Shape: (992, 5)


,user_id,gender,age,country,signup_date
0,user_000001,m,NaN,Japan,"Aug 13, 2006"
1,user_000002,f,NaN,Peru,"Feb 24, 2006"
2,user_000003,m,22,United States,"Oct 30, 2005"
3,user_000004,f,NaN,NaN,"Apr 26, 2006"
4,user_000005,m,NaN,Bulgaria,"Jun 29, 2006"


In [4]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
users["age"] = pd.to_numeric(users["age"], errors="coerce")

print("Rows with unparseable signup date:", users["signup_date"].isna().sum(), "out of", len(users))
users.head()

Rows with unparseable signup date: 8 out of 992


,user_id,gender,age,country,signup_date
0,user_000001,m,NaN,Japan,2006-08-13
1,user_000002,f,NaN,Peru,2006-02-24
2,user_000003,m,22.0,United States,2005-10-30
3,user_000004,f,NaN,NaN,2006-04-26
4,user_000005,m,NaN,Bulgaria,2006-06-29


In [5]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)   # start fresh each full run
conn = sqlite3.connect(DB_FILE)

col_names = ["user_id", "timestamp", "artist_id", "artist_name", "track_id", "track_name"]
CHUNK_SIZE = 1_000_000

total_read = 0
total_kept = 0
first_chunk = True

reader = pd.read_csv(
    EVENTS_FILE,
    sep="\t",
    header=None,
    names=col_names,
    on_bad_lines="skip",       # drops the small number of malformed rows real data always has
    quoting=csv.QUOTE_NONE,    # track/artist names sometimes contain stray quote characters
    dtype=str,
    encoding="utf-8",
    chunksize=CHUNK_SIZE,
)

for i, chunk in enumerate(reader, start=1):
    before = len(chunk)
    total_read += before

    chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], errors="coerce", utc=True)
    chunk = chunk.dropna(subset=["timestamp", "user_id"])
    chunk["stream_date"] = chunk["timestamp"].dt.date.astype(str)
    chunk["timestamp"] = chunk["timestamp"].astype(str)

    chunk.to_sql("listens", conn, if_exists="replace" if first_chunk else "append", index=False)
    first_chunk = False
    total_kept += len(chunk)

    print(f"Chunk {i}: read {before:,} rows, kept {len(chunk):,} after cleaning | running total: {total_kept:,}")

conn.commit()
print(f"\nDone. Raw rows read: {total_read:,} | Rows loaded into DB: {total_kept:,} | Dropped: {total_read - total_kept:,}")

Chunk 1: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 1,000,000
Chunk 2: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 2,000,000
Chunk 3: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 3,000,000
Chunk 4: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 4,000,000
Chunk 5: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 5,000,000
Chunk 6: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 6,000,000
Chunk 7: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 7,000,000
Chunk 8: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 8,000,000
Chunk 9: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 9,000,000
Chunk 10: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 10,000,000
Chunk 11: read 1,000,000 rows, kept 1,000,000 after cleaning | running total: 11,000,000
Chunk 12: read 1,000,000 rows, kept 1,0

In [6]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_listens_user ON listens(user_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_listens_date ON listens(stream_date)")
conn.commit()

check = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM listens", conn)
print(check)

   total_rows
0    19150868


In [7]:
import numpy as np
np.random.seed(42)

plays_per_user = pd.read_sql_query(
    "SELECT user_id, COUNT(*) AS total_plays FROM listens GROUP BY user_id", conn
)

users = users.merge(plays_per_user, on="user_id", how="left")
users["total_plays"] = users["total_plays"].fillna(0)

quantiles = users["total_plays"].rank(pct=True)
users["plan_type"] = np.select(
    [quantiles < 0.35, quantiles < 0.70, quantiles < 0.90],
    ["Free", "Premium", "Family"],
    default="Student",
)

plan_price = {"Free": 0, "Premium": 9.99, "Family": 14.99, "Student": 4.99}
users["monthly_price"] = users["plan_type"].map(plan_price)

users_for_sql = users.copy()
users_for_sql["signup_date"] = users_for_sql["signup_date"].astype(str)
users_for_sql.to_sql("users", conn, if_exists="replace", index=False)
conn.commit()

print(users["plan_type"].value_counts())
users.head()

plan_type
Premium    347
Free       347
Family     198
Student    100
Name: count, dtype: int64


,user_id,gender,age,country,signup_date,total_plays,plan_type,monthly_price
0,user_000001,m,NaN,Japan,2006-08-13,16685,Premium,9.99
1,user_000002,f,NaN,Peru,2006-02-24,57438,Student,4.99
2,user_000003,m,22.0,United States,2005-10-30,19494,Premium,9.99
3,user_000004,f,NaN,NaN,2006-04-26,18411,Premium,9.99
4,user_000005,m,NaN,Bulgaria,2006-06-29,20341,Premium,9.99


In [8]:
def run(sql):
    return pd.read_sql_query(sql, conn)

print("--- Row counts ---")
print(run("SELECT 'users' AS table_name, COUNT(*) AS n FROM users UNION ALL SELECT 'listens', COUNT(*) FROM listens"))

print("\n--- Date range ---")
print(run("SELECT MIN(stream_date) AS first_listen, MAX(stream_date) AS last_listen FROM listens"))

print("\n--- Null checks ---")
print(run("SELECT SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user, SUM(CASE WHEN track_name IS NULL THEN 1 ELSE 0 END) AS null_track FROM listens"))

print("\n--- Users by (simulated) plan ---")
print(run("SELECT plan_type, COUNT(*) AS num_users FROM users GROUP BY plan_type ORDER BY num_users DESC"))

print("\n--- Users by real country ---")
print(run("SELECT country, COUNT(*) AS num_users FROM users WHERE country IS NOT NULL AND country != '' GROUP BY country ORDER BY num_users DESC LIMIT 10"))

print("\n--- Listens per month (real trend) ---")
print(run("SELECT substr(stream_date,1,7) AS month, COUNT(*) AS total_listens FROM listens GROUP BY month ORDER BY month"))

print("\n--- Top 10 tracks by real play count (preview of Day 3) ---")
print(run("SELECT track_name, artist_name, COUNT(*) AS play_count FROM listens GROUP BY track_name, artist_name ORDER BY play_count DESC LIMIT 10"))

--- Row counts ---
  table_name         n
0      users       992
1    listens  19150868

--- Date range ---
  first_listen last_listen
0   2005-02-14  2013-09-29

--- Null checks ---
   null_user  null_track
0          0         209

--- Users by (simulated) plan ---
  plan_type  num_users
0   Premium        347
1      Free        347
2    Family        198
3   Student        100

--- Users by real country ---
          country  num_users
0   United States        228
1  United Kingdom        126
2          Poland         50
3         Germany         36
4          Norway         35
5         Finland         32
6          Canada         32
7          Turkey         28
8           Italy         27
9          Sweden         24

--- Listens per month (real trend) ---
      month  total_listens
0   2005-02          24690
1   2005-03          49273
2   2005-04          74085
3   2005-05          69549
4   2005-06          81691
5   2005-07          87337
6   2005-08         100275
7   2005-09

In [1]:
import pandas as pd
import sqlite3

DB_FILE = "lastfm_streaming.db"
conn = sqlite3.connect(DB_FILE)

def run(sql):
    return pd.read_sql_query(sql, conn)

print(run("SELECT COUNT(*) AS n FROM listens"))

          n
0  19150868


In [2]:
ANALYSIS_END_DATE = "2009-05-31"

before = run("SELECT COUNT(*) AS n FROM listens").iloc[0]["n"]
after = run(f"SELECT COUNT(*) AS n FROM listens WHERE stream_date > '{ANALYSIS_END_DATE}'").iloc[0]["n"]
print(f"Total rows: {before:,} | Rows after cutoff (excluded from analysis): {after:,}")

Total rows: 19,150,868 | Rows after cutoff (excluded from analysis): 55,599


In [3]:
dau = run(f"""
    SELECT stream_date, COUNT(DISTINCT user_id) AS dau
    FROM listens
    WHERE stream_date <= '{ANALYSIS_END_DATE}'
    GROUP BY stream_date
    ORDER BY stream_date
""")
print(dau.shape)
dau.head()

(1568, 2)


,stream_date,dau
0,2005-02-14,35
1,2005-02-15,33
2,2005-02-16,37
3,2005-02-17,35
4,2005-02-18,33


In [4]:
mau = run(f"""
    SELECT substr(stream_date,1,7) AS month, COUNT(DISTINCT user_id) AS mau
    FROM listens
    WHERE stream_date <= '{ANALYSIS_END_DATE}'
    GROUP BY month
    ORDER BY month
""")
print(mau)

      month  mau
0   2005-02   55
1   2005-03   71
2   2005-04   83
3   2005-05   88
4   2005-06   98
5   2005-07  111
6   2005-08  122
7   2005-09  134
8   2005-10  147
9   2005-11  162
10  2005-12  203
11  2006-01  226
12  2006-02  264
13  2006-03  296
14  2006-04  322
15  2006-05  344
16  2006-06  387
17  2006-07  392
18  2006-08  426
19  2006-09  427
20  2006-10  440
21  2006-11  455
22  2006-12  471
23  2007-01  498
24  2007-02  506
25  2007-03  508
26  2007-04  512
27  2007-05  516
28  2007-06  528
29  2007-07  525
30  2007-08  548
31  2007-09  562
32  2007-10  568
33  2007-11  570
34  2007-12  566
35  2008-01  567
36  2008-02  569
37  2008-03  590
38  2008-04  592
39  2008-05  601
40  2008-06  600
41  2008-07  615
42  2008-08  613
43  2008-09  606
44  2008-10  622
45  2008-11  640
46  2008-12  664
47  2009-01  654
48  2009-02  673
49  2009-03  695
50  2009-04  775
51  2009-05  513


In [5]:
dau["month"] = dau["stream_date"].str.slice(0, 7)
stickiness = dau.groupby("month")["dau"].mean().reset_index().rename(columns={"dau": "avg_dau"})
stickiness = stickiness.merge(mau, on="month")
stickiness["stickiness_pct"] = (100 * stickiness["avg_dau"] / stickiness["mau"]).round(1)
stickiness

,month,avg_dau,mau,stickiness_pct
0,2005-02,35.466667,55,64.5
1,2005-03,39.774194,71,56.0
2,2005-04,48.733333,83,58.7
3,2005-05,46.193548,88,52.5
4,2005-06,53.966667,98,55.1
5,2005-07,54.225806,111,48.9
6,2005-08,64.935484,122,53.2
7,2005-09,74.333333,134,55.5
8,2005-10,87.032258,147,59.2
9,2005-11,98.166667,162,60.6


In [6]:
cohort_sql = f"""
WITH user_cohort AS (
    SELECT user_id, strftime('%Y-%m', signup_date) AS cohort_month
    FROM users
    WHERE signup_date IS NOT NULL AND signup_date != 'NaT'
),
user_activity AS (
    SELECT DISTINCT user_id, substr(stream_date,1,7) AS activity_month
    FROM listens
    WHERE stream_date <= '{ANALYSIS_END_DATE}'
),
cohort_activity AS (
    SELECT
        uc.cohort_month,
        ua.activity_month,
        (CAST(strftime('%Y', ua.activity_month || '-01') AS INT) - CAST(strftime('%Y', uc.cohort_month || '-01') AS INT)) * 12
        + (CAST(strftime('%m', ua.activity_month || '-01') AS INT) - CAST(strftime('%m', uc.cohort_month || '-01') AS INT)) AS month_number,
        ua.user_id
    FROM user_cohort uc
    JOIN user_activity ua ON uc.user_id = ua.user_id
    WHERE ua.activity_month >= uc.cohort_month
),
cohort_size AS (
    SELECT cohort_month, COUNT(*) AS num_users FROM user_cohort GROUP BY cohort_month
)
SELECT
    ca.cohort_month, ca.month_number,
    COUNT(DISTINCT ca.user_id) AS retained_users,
    cs.num_users AS cohort_size,
    ROUND(100.0 * COUNT(DISTINCT ca.user_id) / cs.num_users, 1) AS retention_pct
FROM cohort_activity ca
JOIN cohort_size cs ON ca.cohort_month = cs.cohort_month
WHERE ca.month_number BETWEEN 0 AND 12
GROUP BY ca.cohort_month, ca.month_number
ORDER BY ca.cohort_month, ca.month_number
"""
cohort = run(cohort_sql)
print(cohort.shape)
cohort.head(15)

(521, 5)


,cohort_month,month_number,retained_users,cohort_size,retention_pct
0,2004-04,10,4,12,33.3
1,2004-04,11,6,12,50.0
2,2004-04,12,6,12,50.0
3,2004-05,9,2,5,40.0
4,2004-05,10,2,5,40.0
5,2004-05,11,2,5,40.0
6,2004-05,12,2,5,40.0
7,2004-07,7,1,4,25.0
8,2004-07,8,1,4,25.0
9,2004-07,9,1,4,25.0


In [7]:
cohort.to_csv("cohort_retention_real.csv", index=False)
mau.to_csv("mau_real.csv", index=False)
print("Saved cohort_retention_real.csv and mau_real.csv to your project folder.")

Saved cohort_retention_real.csv and mau_real.csv to your project folder.


In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("lastfm_streaming.db")
ANALYSIS_END_DATE = "2009-05-31"   

def run(sql):
    return pd.read_sql_query(sql, conn)

In [2]:
churn = run(f"""
WITH filtered AS (
    SELECT * FROM listens WHERE stream_date <= '{ANALYSIS_END_DATE}'
),
last_activity AS (
    SELECT user_id, MAX(stream_date) AS last_stream_date FROM filtered GROUP BY user_id
),
max_date AS (
    SELECT MAX(stream_date) AS dataset_max_date FROM filtered
),
churn_flag AS (
    SELECT
        u.user_id, u.plan_type,
        CASE
            WHEN la.last_stream_date IS NULL THEN 1
            WHEN julianday((SELECT dataset_max_date FROM max_date)) - julianday(la.last_stream_date) > 60 THEN 1
            ELSE 0
        END AS is_churned
    FROM users u
    LEFT JOIN last_activity la ON u.user_id = la.user_id
)
SELECT plan_type, COUNT(*) AS total_users, SUM(is_churned) AS churned_users,
       ROUND(100.0 * SUM(is_churned) / COUNT(*), 1) AS churn_rate_pct
FROM churn_flag
GROUP BY plan_type
ORDER BY churn_rate_pct DESC
""")
churn

,plan_type,total_users,churned_users,churn_rate_pct
0,Free,347,89,25.6
1,Premium,347,30,8.6
2,Family,198,13,6.6
3,Student,100,2,2.0


In [3]:
top_songs = run(f"""
SELECT track_name, artist_name, COUNT(*) AS play_count,
       COUNT(DISTINCT user_id) AS unique_listeners
FROM listens
WHERE stream_date <= '{ANALYSIS_END_DATE}'
GROUP BY track_name, artist_name
ORDER BY play_count DESC
LIMIT 20
""")
top_songs

,track_name,artist_name,play_count,unique_listeners
0,Such Great Heights,The Postal Service,3990,321
1,Love Will Tear Us Apart,Boy Division,3660,318
2,Karma Police,Radiohead,3532,346
3,Soul Meets Body,Death Cab For Cutie,3469,233
4,Supermassive Black Hole,Muse,3469,262
5,Heartbeats,The Knife,3154,176
6,Starlight,Muse,3051,260
7,Rebellion (Lies),Arcade Fire,3047,292
8,Gimme More,Britney Spears,3002,59
9,When You Were Young,The Killers,2998,235


In [4]:
top_by_country = run(f"""
WITH ranked AS (
    SELECT u.country, l.track_name, l.artist_name, COUNT(*) AS play_count,
           RANK() OVER (PARTITION BY u.country ORDER BY COUNT(*) DESC) AS country_rank
    FROM listens l
    JOIN users u ON l.user_id = u.user_id
    WHERE l.stream_date <= '{ANALYSIS_END_DATE}' AND u.country IS NOT NULL AND u.country != ''
    GROUP BY u.country, l.track_name, l.artist_name
)
SELECT * FROM ranked WHERE country_rank <= 5
ORDER BY country, country_rank
""")
top_by_country.head(20)

,country,track_name,artist_name,play_count,country_rank
0,Algeria,Needle In The Hay,Elliott Smith,26,1
1,Algeria,Clementine,Elliott Smith,20,2
2,Algeria,Before And Again,Akron/Family,19,3
3,Algeria,Single File,Elliott Smith,19,3
4,Algeria,Christian Brothers,Elliott Smith,18,5
5,Algeria,Coming Up Roses,Elliott Smith,18,5
6,Algeria,Southern Belle,Elliott Smith,18,5
7,Algeria,Suchness,Akron/Family,18,5
8,Antarctica,Knights Of Cydonia,Muse,86,1
9,Antarctica,Pass This On,The Knife,75,2


In [5]:
top_songs.to_csv("top_songs_real.csv", index=False)
top_by_country.to_csv("top_songs_by_country_real.csv", index=False)
print("Saved.")

Saved.


In [6]:
revenue = run(f"""
WITH active_months AS (
    SELECT DISTINCT user_id, substr(stream_date,1,7) AS month
    FROM listens
    WHERE stream_date <= '{ANALYSIS_END_DATE}'
)
SELECT am.month, u.plan_type, ROUND(SUM(u.monthly_price), 2) AS revenue
FROM active_months am
JOIN users u ON am.user_id = u.user_id
WHERE u.plan_type != 'Free'
GROUP BY am.month, u.plan_type
ORDER BY am.month
""")
revenue

,month,plan_type,revenue
0,2005-02,Family,284.81
1,2005-02,Premium,179.82
2,2005-02,Student,59.88
3,2005-03,Family,329.78
4,2005-03,Premium,219.78
...,...,...,...
151,2009-04,Premium,2907.09
152,2009-04,Student,479.04
153,2009-05,Family,1618.92
154,2009-05,Premium,1988.01


In [7]:
total_revenue = run(f"""
WITH active_months AS (
    SELECT DISTINCT user_id, substr(stream_date,1,7) AS month
    FROM listens
    WHERE stream_date <= '{ANALYSIS_END_DATE}'
)
SELECT am.month, ROUND(SUM(u.monthly_price), 2) AS total_revenue
FROM active_months am
JOIN users u ON am.user_id = u.user_id
WHERE u.plan_type != 'Free'
GROUP BY am.month
ORDER BY am.month
""")

arpu = run("""
SELECT plan_type, monthly_price, COUNT(*) AS num_users,
       ROUND(monthly_price * COUNT(*), 2) AS monthly_revenue_at_full_activity
FROM users
GROUP BY plan_type, monthly_price
ORDER BY monthly_revenue_at_full_activity DESC
""")

total_revenue.to_csv("monthly_revenue_real.csv", index=False)
print(total_revenue.tail())
print()
print(arpu)

      month  total_revenue
47  2009-01        5664.70
48  2009-02        5649.72
49  2009-03        5754.62
50  2009-04        6084.33
51  2009-05        3861.42

  plan_type  monthly_price  num_users  monthly_revenue_at_full_activity
0   Premium           9.99        347                           3466.53
1    Family          14.99        198                           2968.02
2   Student           4.99        100                            499.00
3      Free           0.00        347                              0.00


In [8]:
churn.to_csv("churn_by_plan_real.csv", index=False)
print("Saved.")

Saved.


In [9]:
pivot_source = run(f"""
SELECT u.plan_type, u.country, substr(l.stream_date,1,7) AS month,
       COUNT(*) AS play_count, COUNT(DISTINCT l.user_id) AS unique_listeners
FROM listens l
JOIN users u ON l.user_id = u.user_id
WHERE l.stream_date <= '{ANALYSIS_END_DATE}' AND u.country IS NOT NULL AND u.country != ''
GROUP BY u.plan_type, u.country, month
""")
pivot_source.to_csv("pivot_source_real.csv", index=False)
print(pivot_source.shape)

(5457, 5)


In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("lastfm_streaming.db")
ANALYSIS_END_DATE = "2009-05-31"

def run(sql):
    return pd.read_sql_query(sql, conn)

mau = pd.read_csv("mau_real.csv")
cohort = pd.read_csv("cohort_retention_real.csv")
churn = pd.read_csv("churn_by_plan_real.csv")
top_songs = pd.read_csv("top_songs_real.csv")
total_revenue = pd.read_csv("monthly_revenue_real.csv")
pivot_source = pd.read_csv("pivot_source_real.csv")

print("mau:", mau.shape)
print("cohort:", cohort.shape)
print("churn:", churn.shape)
print("top_songs:", top_songs.shape)
print("total_revenue:", total_revenue.shape)
print("pivot_source:", pivot_source.shape)

mau: (52, 2)
cohort: (521, 5)
churn: (4, 4)
top_songs: (20, 4)
total_revenue: (52, 2)
pivot_source: (5457, 5)


In [2]:

import openpyxl
print(openpyxl.__version__)

3.1.5


In [5]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import LineChart, BarChart, Reference
from openpyxl.utils import get_column_letter

FONT = "Arial"
HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(name=FONT, bold=True, color="FFFFFF", size=11)
TITLE_FONT = Font(name=FONT, bold=True, size=14, color="1F4E78")
BODY_FONT = Font(name=FONT, size=10)
THIN = Side(style="thin", color="D9D9D9")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def style_header_row(ws, row, ncols):
    for c in range(1, ncols + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = BORDER

def write_df(ws, df, start_row=1, start_col=1):
    for j, col in enumerate(df.columns):
        ws.cell(row=start_row, column=start_col + j, value=col)
    for i, row in enumerate(df.itertuples(index=False)):
        for j, val in enumerate(row):
            ws.cell(row=start_row + 1 + i, column=start_col + j, value=val).font = BODY_FONT
    style_header_row(ws, start_row, len(df.columns))
    for j, col in enumerate(df.columns):
        col_letter = get_column_letter(start_col + j)
        maxlen = max([len(str(col))] + [len(str(v)) for v in df[col].astype(str).values[:200]])
        ws.column_dimensions[col_letter].width = min(max(maxlen + 2, 10), 40)
    return start_row + 1 + len(df)

wb = Workbook()
ws = wb.active
ws.title = "README"
ws["A1"] = "Music Streaming Analytics — Real Data (Last.fm 1K Users)"
ws["A1"].font = TITLE_FONT
ws["A3"] = "DAU/MAU, cohort retention, and Top Songs are 100% real listening data. plan_type/monthly_price/revenue are a simulated layer on top of real listening volume (see Day 1-3 notes)."
ws["A3"].alignment = Alignment(wrap_text=True)
ws.merge_cells("A3:H3")
ws.row_dimensions[3].height = 30
ws.column_dimensions["A"].width = 20
ws.column_dimensions["B"].width = 90

ws2 = wb.create_sheet("Pivot_Source")
last_row = write_df(ws2, pivot_source)
ws2.freeze_panes = "A2"

ws3 = wb.create_sheet("Pivot_Summary")
ws3["A1"] = "Plays by Plan Type (SUMIFS over Pivot_Source)"
ws3["A1"].font = TITLE_FONT
ws3["A3"] = "plan_type"; ws3["B3"] = "total_plays"
style_header_row(ws3, 3, 2)
plans = sorted(pivot_source["plan_type"].unique().tolist())
for i, p in enumerate(plans):
    row = 4 + i
    ws3.cell(row=row, column=1, value=p).font = BODY_FONT
    ws3.cell(row=row, column=2, value=f"=SUMIFS(Pivot_Source!D2:D{last_row},Pivot_Source!A2:A{last_row},A{row})").font = BODY_FONT

print("README, Pivot_Source, Pivot_Summary built.")

README, Pivot_Source, Pivot_Summary built.


In [6]:
cohort_pivot = cohort.pivot(index="cohort_month", columns="month_number", values="retention_pct")
cohort_pivot.columns = [f"M{c}" for c in cohort_pivot.columns]
cohort_pivot = cohort_pivot.reset_index()

ws5 = wb.create_sheet("Retention_Heatmap")
ws5["A1"] = "Cohort Retention Heatmap (Real)"
ws5["A1"].font = TITLE_FONT
last5 = write_df(ws5, cohort_pivot, start_row=3)

ncols = len(cohort_pivot.columns)
data_range = f"B4:{get_column_letter(ncols)}{last5-1}"
rule = ColorScaleRule(start_type="min", start_color="F8696B", mid_type="percentile", mid_value=50, mid_color="FFEB84", end_type="max", end_color="63BE7B")
ws5.conditional_formatting.add(data_range, rule)
ws5.freeze_panes = "B4"
print("Retention_Heatmap sheet built.")

Retention_Heatmap sheet built.


In [7]:
ws6 = wb.create_sheet("Churn_by_Plan")
ws6["A1"] = "Churn Rate by Plan (Simulated Plan, Real Activity)"
ws6["A1"].font = TITLE_FONT
last6 = write_df(ws6, churn, start_row=3)

bar = BarChart()
bar.title = "Churn Rate by Plan (%)"
data = Reference(ws6, min_col=4, min_row=3, max_row=last6-1)
cats = Reference(ws6, min_col=1, min_row=4, max_row=last6-1)
bar.add_data(data, titles_from_data=True)
bar.set_categories(cats)
ws6.add_chart(bar, "A12")
print("Churn_by_Plan sheet built.")

Churn_by_Plan sheet built.


In [8]:
ws7 = wb.create_sheet("Top_Songs")
ws7["A1"] = "Top Songs by Real Play Count"
ws7["A1"].font = TITLE_FONT
last7 = write_df(ws7, top_songs, start_row=3)

bar2 = BarChart()
bar2.type = "bar"
bar2.title = "Top 10 Songs by Plays"
data = Reference(ws7, min_col=3, min_row=3, max_row=13)
cats = Reference(ws7, min_col=1, min_row=4, max_row=13)
bar2.add_data(data, titles_from_data=True)
bar2.set_categories(cats)
ws7.add_chart(bar2, "F3")
print("Top_Songs sheet built.")

Top_Songs sheet built.


In [9]:
ws8 = wb.create_sheet("Revenue_Forecast")
ws8["A1"] = "Monthly Revenue — History & Forecast (Simulated Revenue)"
ws8["A1"].font = TITLE_FONT
ws8["A3"] = "month"; ws8["B3"] = "total_revenue"; ws8["C3"] = "period_number"; ws8["D3"] = "forecast"
style_header_row(ws8, 3, 4)

n_hist = len(total_revenue)
for i, row_ in total_revenue.iterrows():
    r = 4 + i
    ws8.cell(row=r, column=1, value=row_["month"]).font = BODY_FONT
    ws8.cell(row=r, column=2, value=float(row_["total_revenue"])).font = BODY_FONT
    ws8.cell(row=r, column=3, value=i + 1).font = BODY_FONT

hist_last_row = 3 + n_hist
ws8["F1"] = "slope"; ws8["G1"] = f"=SLOPE(B4:B{hist_last_row},C4:C{hist_last_row})"
ws8["F2"] = "intercept"; ws8["G2"] = f"=INTERCEPT(B4:B{hist_last_row},C4:C{hist_last_row})"

for i in range(n_hist):
    r = 4 + i
    ws8.cell(row=r, column=4, value=f"=$G$2+$G$1*C{r}").font = BODY_FONT

last_month = total_revenue["month"].iloc[-1]
ly, lm = map(int, last_month.split("-"))
for k in range(1, 7):
    mo = lm + k
    yr = ly + (mo - 1) // 12
    mo = ((mo - 1) % 12) + 1
    r = hist_last_row + k
    ws8.cell(row=r, column=1, value=f"{yr}-{mo:02d}")
    ws8.cell(row=r, column=3, value=n_hist + k)
    ws8.cell(row=r, column=4, value=f"=$G$2+$G$1*C{r}").font = Font(name=FONT, bold=True, color="C00000")

line = LineChart()
line.title = "Revenue: Actual vs Forecast"
data = Reference(ws8, min_col=2, min_row=3, max_row=hist_last_row)
data2 = Reference(ws8, min_col=4, min_row=3, max_row=hist_last_row + 6)
cats = Reference(ws8, min_col=1, min_row=4, max_row=hist_last_row + 6)
line.add_data(data, titles_from_data=True)
line.add_data(data2, titles_from_data=True)
line.set_categories(cats)
ws8.add_chart(line, "A" + str(hist_last_row + 8))

wb.save("Music_Streaming_Dashboard_Real.xlsx")
print("Workbook saved: Music_Streaming_Dashboard_Real.xlsx")

Workbook saved: Music_Streaming_Dashboard_Real.xlsx


In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

FILE = "Music_Streaming_Dashboard_Real.xlsx"
FONT = "Arial"

wb = load_workbook(FILE)
ws8 = wb["Revenue_Forecast"]

row = 4
hist_last_row = 3
while ws8.cell(row=row, column=2).value is not None:
    hist_last_row = row
    row += 1

print("Detected historical data ends at row:", hist_last_row)

RECENT_MONTHS = 12
recent_start_row = hist_last_row - RECENT_MONTHS + 1

ws8["F4"] = "recent_slope"
ws8["G4"] = f"=SLOPE(B{recent_start_row}:B{hist_last_row},C{recent_start_row}:C{hist_last_row})"
ws8["F5"] = "recent_intercept"
ws8["G5"] = f"=INTERCEPT(B{recent_start_row}:B{hist_last_row},C{recent_start_row}:C{hist_last_row})"

ws8["E3"] = "forecast_recent_trend"
for k in range(1, 7):
    r = hist_last_row + k
    ws8.cell(row=r, column=5, value=f"=$G$5+$G$4*C{r}").font = Font(name=FONT, bold=True, color="2E7D32")

wb.save(FILE)
print("Saved. Column E now has a second forecast based on just the last 12 real months.")

NameError: name 'ws8' is not defined